# Collaborative Memory | Agent Memory System

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import TypedDict, Dict, List
from threading import Lock

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Collaborative Memory for Multi-Agent Team
# For concurrent writes, implement conflict resolution (last-writer-wins, merge, or version vectors)

class SharedMemory:
    def __init__(self):
        self._store: Dict[str, List[str]] = {}
        self._lock = Lock()

    def write(self, agent: str, key: str, value: str):
        with self._lock:
            self._store.setdefault(key, []).append(f"[{agent}]: {value}")

    def read(self, key: str) -> List[str]:
        with self._lock:
            return list(self._store.get(key, []))

    def read_all(self) -> Dict[str, List[str]]:
        with self._lock:
            return dict(self._store)

shared = SharedMemory()

In [4]:
class TeamState(TypedDict):
    topic: str
    final_report: str

def researcher(state: TeamState) -> dict:
    response = model.invoke(f"Research this topic:\n\n{state['topic']}")
    shared.write("researcher", "findings", response.content)
    return {}

def analyst(state: TeamState) -> dict:
    findings = shared.read("findings")
    response = model.invoke(f"Analyze these findings:\n\n{''.join(findings)}")
    shared.write("analyst", "analysis", response.content)
    return {}

def writer(state: TeamState) -> dict:
    all_mem = shared.read_all()
    context = "\n\n".join(f"## {k}\n" + "\n".join(v) for k, v in all_mem.items())
    response = model.invoke(f"Write a report from team contributions:\n\n{context}")
    return {"final_report": response.content}

graph = StateGraph(TeamState)
graph.add_sequence([("researcher", researcher), ("analyst", analyst), ("writer", writer)])
graph.add_edge(START, "researcher")
graph.add_edge("writer", END)

team = graph.compile()
result = team.invoke({"topic": "Impact of AI on healthcare diagnostics"})
print(result["final_report"])

# Report on AI's Impact on Healthcare Diagnostics

## Team Contributions Overview

In examining the role of AI in healthcare diagnostics, our research and analysis have yielded a comprehensive understanding of its profound impact across various dimensions of healthcare. Here's a detailed overview of the team's findings and analysis:

### Key Findings

1. **Improved Diagnostic Accuracy**: 
   AI algorithms, particularly deep learning models, have demonstrated superior accuracy in diagnosing diseases from medical images like X-rays, MRIs, and CT scans. These models can identify intricate patterns that may not be detected by human observation, which has the potential to significantly reduce diagnostic errors and misdiagnoses.

2. **Early Detection and Prevention**: 
   AI systems excel in rapidly analyzing large datasets to identify trends and risk factors, leading to earlier detection of serious conditions such as cancer and cardiovascular diseases. This early diagnosis allows for prompt